In [36]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()

,ID,Age,Race,Marital Status,T Stage,N Stage,6th Stage,differentiate,Grade,A Stage,...,BMI,Heart Rate,Serum Creatinine,Uric Acid,Hemoglobin,GFR,Serum Sodium,Serum Potassium,Serum Albumin,Lactate
0,1,62.300654,White,Widowed,T1,N1,NaN,Moderately differentiated,2,Regional,...,25.228195,98.865449,4.343779,3.781804,13.704187,112.038394,136.560377,-1.506035,4.699045,-3.200633
1,2,37.268422,White,Married,T2,N1,IIB,Well differentiated,1,Regional,...,31.027525,81.547091,-5.053593,1.990754,20.685675,109.605432,147.569841,-0.033068,1.676842,2.013738
2,3,55.864953,White,Single,T1,N1,IIA,Well differentiated,1,Regional,...,20.009729,77.214648,-0.683623,11.299137,10.565341,112.964603,147.176105,-7.076607,6.776799,-7.266369
3,4,60.586799,White,Divorced,T1,N1,IIA,Poorly differentiated,3,Regional,...,33.217567,86.513469,0.317514,0.238220,7.512031,63.477023,135.443017,3.189928,4.519103,-1.637100
4,5,48.197741,White,Separated,T2,N1,IIB,Moderately differentiated,2,Regional,...,33.909838,90.401178,-1.838835,10.500072,19.806571,98.437718,136.071561,8.390279,7.027023,11.230639


In [31]:
# T1, T2, T3, T4
bins1 = [-float('inf'), 90, float('inf')]
labels1 = ['Mildly Decreased', 'Normal']
test['T1'] = pd.cut(test['GFR'], bins=bins1, labels=labels1, right=False)

Q1 = train['Serum Creatinine'].quantile(.25)
Q2 = train['Serum Creatinine'].quantile(.5)
Q3 = train['Serum Creatinine'].quantile(.75)

bins2 = [-float('inf'), Q1, Q2, Q3, float('inf')]
labels2 = ['Very Low', 'Low', 'High', 'Very High']

test['T2'] = pd.cut(test['Serum Creatinine'], bins=bins2, labels=labels2, right=True)
med = train['BMI'].median()
bins3 = [-float('inf'), med, float('inf')]
labels3 = [0, 1]
test['T3'] = pd.cut(test['BMI'], bins=bins3, labels=labels3, right=False)
test['T4'] = test['T Stage'].apply(lambda x: (train['T Stage'] == x).sum())

task1 = test['T1']
task2 = test['T2']
task3 = test['T3']
task4 = test['T4']
test = test.drop(columns=['T1', 'T2', 'T3', 'T4'])

In [39]:
le = LabelEncoder()
y_train = le.fit_transform(train['Status'])
x_train = train.drop(columns=['ID', 'Status'])
x_test = test.drop(columns=['ID'])

objCols = list(x_train.select_dtypes(include='str').columns)

preprocessor = ColumnTransformer(
    transformers=[
        ('oneHot', OneHotEncoder(drop='first', handle_unknown='ignore'), objCols)
    ],
    remainder='passthrough'
)
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', HistGradientBoostingClassifier(
        random_state=42,
        max_iter=500,
        max_depth=6,
        min_samples_leaf=20,
        l2_regularization=0.1,
        learning_rate=.01
    ))
])
param_dist = {
    'model__max_iter':         [200, 500],
    'model__learning_rate':    [0.05, 0.1],
    'model__max_depth':        [4, 6],
    'model__min_samples_leaf': [10, 20, 40],
}
rs = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=6,
    scoring='precision',
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=2
)

rs.fit(x_train, y_train)
model = rs.best_estimator_
predictions = model.predict(x_test)


Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END model__learning_rate=0.05, model__max_depth=4, model__max_iter=200, model__min_samples_leaf=10; total time=   0.6s
[CV] END model__learning_rate=0.05, model__max_depth=6, model__max_iter=200, model__min_samples_leaf=40; total time=   0.7s
[CV] END model__learning_rate=0.05, model__max_depth=4, model__max_iter=200, model__min_samples_leaf=10; total time=   0.9s
[CV] END model__learning_rate=0.05, model__max_depth=4, model__max_iter=200, model__min_samples_leaf=10; total time=   1.0s
[CV] END model__learning_rate=0.05, model__max_depth=6, model__max_iter=200, model__min_samples_leaf=40; total time=   1.2s
[CV] END model__learning_rate=0.05, model__max_depth=6, model__max_iter=200, model__min_samples_leaf=40; total time=   1.3s
[CV] END model__learning_rate=0.1, model__max_depth=6, model__max_iter=200, model__min_samples_leaf=10; total time=   1.3s
[CV] END model__learning_rate=0.1, model__max_depth=6, model__max_iter=20

In [45]:
rows = []
predictions = le.inverse_transform(predictions)
for id, t1 ,t2, t3, t4, pred in zip(test['ID'], task1, task2, task3, task4, predictions):
    rows.append({
        'subtaskID': 1,
        'datapointID': id,
        'answer': t1
    })
    rows.append({
        'subtaskID': 2,
        'datapointID': id,
        'answer': t2
    })
    rows.append({
        'subtaskID': 3,
        'datapointID': id,
        'answer': t3
    })
    rows.append({
        'subtaskID': 4,
        'datapointID': id,
        'answer': t4
    })
    rows.append({
        'subtaskID': 5,
        'datapointID': id,
        'answer': pred
    })
sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)